In [0]:
# Configure access to your Azure Storage Account
storage_account_name = "storagebaitaccount"
storage_account_key = "kJwVMSEv8dSOi8KEycrP7TElef63aMGPEta1i51Rs4z/Xlz8Ib98LdU6Qw/7YldQcmEMRQNAeImC+AStxeSXZw=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)

In [0]:
container_name = "raw-data"
file_name = "cities.csv"

file_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/{file_name}"

# Load the data
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# Verify the data
display(df)

city_name,country_code
Paris,FR
Lyon,FR
Marseille,FR
Toulouse,FR
Nice,FR
Nantes,FR
Strasbourg,FR
Montpellier,FR
Bordeaux,FR
Lille,FR


In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp

# 1. Fetch live data
url = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/velib-disponibilite-en-temps-reel/records?limit=100"
response = requests.get(url)
data = response.json().get("results", [])

if data:
    # 2. Convert to Pandas first (it handles the JSON types better)
    pdf = pd.DataFrame(data)
    
    # 3. Convert Pandas to Spark
    # This solves the [CANNOT_DETERMINE_TYPE] error
    df = spark.createDataFrame(pdf).withColumn('ingestion_time', current_timestamp())
    
    # 4. Save to Bronze
    df.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("yes_catalog.bronze.stations_raw")
    
    print(f"Success! {len(data)} rows ingested.")
    display(df)
else:
    print("API returned no data.")

Success! 100 rows ingested.


stationcode,name,is_installed,capacity,numdocksavailable,numbikesavailable,mechanical,ebike,is_renting,is_returning,duedate,coordonnees_geo,nom_arrondissement_communes,code_insee_commune,station_opening_hours,ingestion_time
40001,Hôpital Mondor,OUI,28,22,6,4,2,OUI,OUI,2026-04-29T12:52:42+00:00,"List(48.798922410229, 2.4537451531298)",Créteil,94028,null,2026-04-29T13:03:50.229421Z
32304,Charcot - Benfleet,OUI,28,26,1,0,1,OUI,OUI,2026-04-29T12:54:32+00:00,"List(48.878370277021, 2.440523876268)",Romainville,93063,null,2026-04-29T13:03:50.229421Z
14111,Cassini - Denfert-Rochereau,OUI,25,19,5,1,4,OUI,OUI,2026-04-29T12:54:10+00:00,"List(48.837525839067, 2.3360354080796)",Paris,75056,null,2026-04-29T13:03:50.229421Z
11104,Charonne - Robert et Sonia Delaunay,OUI,20,18,2,0,2,OUI,OUI,2026-04-29T12:51:04+00:00,"List(48.855907555969, 2.3925706744194)",Paris,75056,null,2026-04-29T13:03:50.229421Z
7002,Vaneau - Sèvres,OUI,35,14,19,18,1,OUI,OUI,2026-04-29T12:55:34+00:00,"List(48.848563233059, 2.3204218259346)",Paris,75056,null,2026-04-29T13:03:50.229421Z
5110,Lacépède - Monge,OUI,23,18,5,2,3,OUI,OUI,2026-04-29T12:46:36+00:00,"List(48.84389286531899, 2.3519663885235786)",Paris,75056,null,2026-04-29T13:03:50.229421Z
6108,Saint-Romain - Cherche-Midi,OUI,17,9,6,5,1,OUI,OUI,2026-04-29T12:52:43+00:00,"List(48.84708159081946, 2.321374788880348)",Paris,75056,null,2026-04-29T13:03:50.229421Z
33006,André Karman - République,OUI,31,21,10,4,6,OUI,OUI,2026-04-29T12:54:01+00:00,"List(48.91039875761846, 2.3851355910301213)",Aubervilliers,93001,null,2026-04-29T13:03:50.229421Z
42016,Pierre et Marie Curie - Maurice Thorez,OUI,27,23,3,2,1,OUI,OUI,2026-04-29T12:52:13+00:00,"List(48.81580226360801, 2.376804985105991)",Ivry-sur-Seine,94041,null,2026-04-29T13:03:50.229421Z
30002,Jean Rostand - Paul Vaillant Couturier,OUI,40,32,8,0,8,OUI,OUI,2026-04-29T12:54:04+00:00,"List(48.908168131015, 2.4530601033354)",Bobigny,93008,null,2026-04-29T13:03:50.229421Z


In [0]:
# 1. Load your City list from the CSV we uploaded earlier
cities_df = spark.read.format("csv").option("header", "true").load(file_path)

# 2. Load your raw bike data from the Bronze table
bikes_df = spark.read.table("yes_catalog.bronze.stations_raw")

# 3. Join them! (This proves which stations belong to which city in your list)
# Note: Since the API is only for Paris, we can 'crossJoin' or filter
final_report = bikes_df.crossJoin(cities_df.filter("city_name = 'Paris'"))

display(final_report.select("city_name", "name", "numbikesavailable"))

city_name,name,numbikesavailable
Paris,Hôpital Mondor,6
Paris,Charcot - Benfleet,1
Paris,Cassini - Denfert-Rochereau,5
Paris,Charonne - Robert et Sonia Delaunay,2
Paris,Vaneau - Sèvres,19
Paris,Lacépède - Monge,5
Paris,Saint-Romain - Cherche-Midi,6
Paris,André Karman - République,10
Paris,Pierre et Marie Curie - Maurice Thorez,3
Paris,Jean Rostand - Paul Vaillant Couturier,8
